<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../6.%20advanced_features/16.%20testing_flask_applications.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#8592; Chapter 16: Testing</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#128218; Table of Contents</a>
  <a href='18.%20deployment.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 18: Deployment &#8594;</a>
</div>

# Chapter 17: Caching and Performance -- Making Flask Fast

> *"Your blog gets featured on the front page of a news site. Suddenly 50,000 people load your home page per minute. Each request queries the database, renders templates, and computes data. Without caching, your database collapses. Caching stores computed results so you don't repeat expensive work."*

## 🎯 The Big Picture

**Caching** is the practice of storing the result of an expensive operation so that future requests for the same result can be served faster — from memory rather than recomputing from scratch.

Every web request that hits your Flask app goes through a chain of expensive operations:

1. Parse the HTTP request
2. Run Python code in the route handler
3. Query the database (potentially multiple times)
4. Render a Jinja2 template
5. Serialize the response

When that response would be **identical** for many users, you're repeating all that work needlessly. Caching breaks this cycle.

### Cache Hit vs Cache Miss

- **Cache hit**: The cached value is found → return it immediately. This takes *microseconds*, because there is no database round-trip, no Python logic, no template rendering.
- **Cache miss**: The cached value is not found → execute the full operation (DB query, template render, etc.), store the result in the cache, then return it. The first request for any given URL always misses; every subsequent request (within the TTL window) hits.

```text
Without cache:
  Request 1  ->  DB query  ->  Template render  ->  Response
  Request 2  ->  DB query  ->  Template render  ->  Response
  Request 3  ->  DB query  ->  Template render  ->  Response
  ... 50,000 requests -> database collapses

With cache (5-min TTL):
  Request 1  ->  DB query  ->  Template render  ->  Store result  ->  Response
  Request 2  ->  Cache hit!  ->  Response (1ms)
  Request 3  ->  Cache hit!  ->  Response (1ms)
  ... 49,999 requests -> database barely touched
```

### TTL — Time-To-Live

Every cache entry has a **TTL (Time-To-Live)**: a timer that automatically expires the entry after N seconds. Once expired, the next request triggers a cache miss, the value is refreshed from the database, and the cycle repeats. TTL is your first line of defence against stale data.

| TTL value | Behaviour |
|---|---|
| `0` or `None` | Never expires — data can become permanently stale (dangerous) |
| `60` | Entry lives 60 seconds; auto-refreshed on next request after expiry |
| `300` | 5 minutes — a common default for slowly-changing public pages |

Short TTLs → more database load but fresher data. Long TTLs → less database load but data may be older.

### The Trade-off: Speed vs Freshness

Caching is always a trade-off: the longer your TTL, the faster your app, but the more out-of-date cached data may become. The right TTL depends on how often the underlying data changes and how much staleness your users can tolerate.

> ❓ **When should I *not* cache something?** Avoid caching data that must always be fresh (account balances, real-time prices), per-user private data in a shared cache, or responses containing secrets. Cache only public, slowly-changing data that's expensive to recompute.

## 🧠 Core Concepts: The "Why"

### Caching is a Whiteboard in Your Kitchen

Instead of going to the grocery store (database) every time you need to know what food you have, you write it on the whiteboard (cache) and check there first. If the whiteboard has the answer, great — you skip the trip. If not (cache miss), you go to the store and update the whiteboard.

The critical question: **when do you erase the whiteboard?** (Cache invalidation — widely considered the hardest problem in computer science.)

### Cache Eviction Policies

What happens when the cache fills up, or entries age out? That's governed by the **eviction policy**:

**LRU — Least Recently Used**
When the cache reaches its capacity limit, it must discard *something* to make room for a new entry. LRU discards the entry that has gone the longest without being accessed — the "coldest" data. Example: your cache holds 1,000 entries; when entry 1,001 arrives, the entry that was read least recently is evicted. This keeps frequently-accessed ("hot") data alive and clears out data nobody is using.

**TTL-based expiry**
Every entry has a maximum age (e.g. 300 seconds). After it expires, the next request triggers a cache miss and the value is refreshed from the source of truth. TTL-based expiry is independent of LRU — an entry can be evicted by TTL before the cache is full. This ensures data freshness: the cache can *never* serve data older than the configured TTL.

In practice, Flask-Caching combines both: LRU eviction when the cache is full, and TTL expiry so entries don't live forever regardless of access patterns.

### Where Caching Helps Most

| Scenario | Cache value | Cache duration |
|----------|-------------|----------------|
| Home page (popular posts) | Rendered HTML | 5 minutes |
| User profile page | HTML fragment | 10 minutes |
| Database query results | Python objects | 60 seconds |
| Navigation menu | HTML | 30 minutes |
| API response | JSON | 2 minutes |

### Where Caching Hurts

- **User-specific data** cached without user variation = data leaks
- **Never-invalidated caches** = stale data users never see updates
- **Caching after a write** without invalidating = old data persists

## ⚡ Syntax & First Usage

### Flask-Caching Setup

**Flask-Caching** is a Flask extension that wraps multiple cache backends behind a unified API. You configure a backend once, then use the same decorators (`@cache.cached()`, `@cache.memoize()`) regardless of whether the underlying store is SimpleCache, Redis, or Memcached. Switching backends in production is a one-line config change — your application code stays the same.

#### Backend Options

**SimpleCache** (in-memory, inside your Python process):
- Fast — no network round trip, no external service required
- Data is lost when the process restarts or is recycled
- **Not shared between processes** — each Gunicorn worker has its own isolated cache instance
- ✅ Good for: development, testing, single-process deployments

**RedisCache** (external Redis server):
- Data survives app restarts (until Redis evicts it by its own policy)
- **Shared across all Gunicorn workers and processes** — worker 1's cached value is available to workers 2, 3, and 4 immediately
- Requires a running Redis instance (e.g. `redis://localhost:6379`)
- ✅ Good for: any multi-worker production deployment

**MemcachedCache** (external Memcached server):
- Like Redis but purely in-memory with no persistence — pure cross-process sharing without a durable store
- A valid alternative to Redis when you don't need cache persistence across Redis restarts

#### `CACHE_DEFAULT_TIMEOUT`

`CACHE_DEFAULT_TIMEOUT` sets the default TTL (in seconds) applied to every cache entry that does not specify its own `timeout` parameter. If you write `@cache.cached()` without a `timeout=` argument, this config value is used. Set it to `0` to make entries never expire by default — not recommended in production, where unbounded staleness can corrupt user-facing data.

#### Initialisation Patterns

Flask-Caching supports both direct initialisation and the `init_app` pattern:

```python
# Direct initialisation (simple apps):
cache = Cache(app, config={"CACHE_TYPE": "SimpleCache", "CACHE_DEFAULT_TIMEOUT": 300})

# init_app pattern (application factory — preferred for larger apps):
cache = Cache()   # no app bound yet

def create_app():
    app = Flask(__name__)
    app.config["CACHE_TYPE"] = "SimpleCache"
    app.config["CACHE_DEFAULT_TIMEOUT"] = 300
    cache.init_app(app)   # bind cache to this specific app instance
    return app
```

The `init_app` pattern decouples the `Cache` object from a specific app instance, making it straightforward to test with different configs or run multiple app instances in the same process.

In [ ]:

# Install: pip install Flask-Caching
# For Redis: pip install Flask-Caching redis

setup_code = """
from flask import Flask
from flask_caching import Cache

app = Flask(__name__)

# Option 1: SimpleCache -- in-memory, single server, no extra services needed
app.config["CACHE_TYPE"] = "SimpleCache"
app.config["CACHE_DEFAULT_TIMEOUT"] = 300   # 5 minutes default

cache = Cache(app)

# Option 2: Redis cache -- distributed, survives restarts, shared across workers
app.config["CACHE_TYPE"] = "RedisCache"
app.config["CACHE_REDIS_URL"] = "redis://localhost:6379/0"
app.config["CACHE_DEFAULT_TIMEOUT"] = 300

cache = Cache(app)

# Option 3: App factory pattern (init_app)
cache = Cache()   # global, without app

def create_app(config_name="default"):
    app = Flask(__name__)
    app.config["CACHE_TYPE"] = "SimpleCache"
    cache.init_app(app)
    return app
"""

print("Flask-Caching setup options:")
print(setup_code)


### `@cache.cached()` — Route-Level Caching

`@cache.cached()` stores the **entire serialised HTTP response** — status code, headers, and body — for a view function. Subsequent requests within the TTL window return the cached response without executing the view body, hitting the database, or doing any computation.

#### How the Cache Key Is Generated

By default, the cache key is derived from the **request URL path** (and optionally the query string). A request to `/popular` generates the key `"view//popular"`. Two requests for the same URL always hit the same cache entry, regardless of which Gunicorn worker handles them.

You can customise the key with the `key_prefix` parameter:

```python
@cache.cached(timeout=300, key_prefix="homepage")          # fixed key: "homepage"
@cache.cached(timeout=300, key_prefix=lambda: f"page_{request.path}")  # dynamic
```

#### ⚠️ Important: This Caches for ALL Users

`@cache.cached()` caches the response once and serves it to **everyone** who requests the same URL. This means:

- ✅ Safe for public pages where every visitor sees identical content (homepage, category pages, static marketing pages)
- ❌ **Never** use it on views that return user-specific data (dashboards, account pages, personalised feeds) without customising the key to vary per user

If user A visits `/dashboard` first and the response is cached, user B will receive user A's data. This is a **serious privacy and security bug** — not just a performance issue.

#### The `timeout` Parameter

```python
@cache.cached(timeout=300)   # expires in 300 seconds (5 minutes)
@cache.cached(timeout=60)    # expires in 60 seconds
@cache.cached(timeout=None)  # never expires — dangerous in production
@cache.cached()              # uses app.config["CACHE_DEFAULT_TIMEOUT"]
```

Setting `timeout=None` means the entry lives until explicitly deleted or the cache is cleared. In production this risks permanently stale data after the underlying records change.

Subsequent requests return the cached response without executing the view body, hitting the database, or doing any computation:

In [ ]:

route_caching = """
from flask_caching import Cache

cache = Cache(app, config={"CACHE_TYPE": "SimpleCache"})

@app.route("/popular-posts")
@cache.cached(timeout=300)   # cache the entire response for 5 minutes
def popular_posts():
    # This entire function -- including the DB query and template render --
    # is skipped on cache hits. Only runs once every 5 minutes.
    posts = Post.query.order_by(Post.view_count.desc()).limit(10).all()
    return render_template("popular.html", posts=posts)


@app.route("/")
@cache.cached(timeout=60)    # homepage cached for 1 minute
def index():
    latest_posts  = Post.query.order_by(Post.created_at.desc()).limit(5).all()
    popular_posts = Post.query.order_by(Post.view_count.desc()).limit(5).all()
    return render_template("index.html",
        latest_posts=latest_posts,
        popular_posts=popular_posts
    )


# NOTE: Do NOT cache user-specific pages without varying by user!
# @cache.cached()  # WRONG -- all users see the same profile!
# def my_profile():
#     return render_template("profile.html", user=current_user)
"""

print("Route-level caching:")
print(route_caching)
print()
print("How it works:")
print("  1. Request arrives for /popular-posts")
print("  2. Flask checks cache for key 'view//popular-posts'")
print("  3. Cache HIT: return stored bytes immediately (microseconds)")
print("  4. Cache MISS: run function, store result, return response")


### `@cache.memoize()` — Function-Level Caching

`@cache.memoize()` caches the **return value of any Python function** — not just route handlers. The cache key is automatically generated from the **function name AND all its arguments**. Each unique combination of arguments gets its own independent cache entry.

#### Cache Key Generation

The key encodes both the function identity and the serialised arguments:

```text
get_user_posts(user_id=42)   →  cache key: "myapp.views.get_user_posts_42"
get_user_posts(user_id=99)   →  cache key: "myapp.views.get_user_posts_99"
get_popular_posts(limit=10)  →  cache key: "myapp.views.get_popular_posts_10"
get_popular_posts(limit=5)   →  cache key: "myapp.views.get_popular_posts_5"
```

Changing *any* argument creates a new, separate cache entry. This makes `@cache.memoize()` ideal for functions called from multiple places with different inputs.

#### `@cache.memoize()` vs `@cache.cached()`

| | `@cache.cached()` | `@cache.memoize()` |
|---|---|---|
| **What it caches** | Full HTTP response (status + headers + body) | Function return value (any Python object) |
| **Cache key** | Request URL path | Function name + all arguments |
| **Applied to** | Route handler functions | Any Python function |
| **One entry per** | Unique URL | Unique argument combination |
| **Invalidate with** | `cache.delete("view//path")` | `cache.delete_memoized(fn, arg1, arg2)` |

Use `@cache.memoize()` for functions called from multiple places with different arguments. Use `@cache.cached()` for route handlers that serve the same response to all visitors of a given URL.

#### Common Use Case

Database query functions called from both web routes and API endpoints are ideal candidates:

```python
@cache.memoize(timeout=120)
def get_user_posts(user_id, page=1):
    # Cached per (user_id, page) — each combination is independent
    return Post.query.filter_by(author_id=user_id).paginate(page=page)
```

Calling `get_user_posts(42, 1)` caches that result. Calling `get_user_posts(42, 2)` generates a separate entry for page 2. Calling `get_user_posts(99, 1)` creates yet another entry for user 99. Each entry is independently cached and independently invalidatable with `cache.delete_memoized(get_user_posts, 99, 1)`.

Use it for expensive computations or database queries called from multiple places:

In [ ]:

memoize_code = """
# @cache.memoize() caches based on the function's arguments
# Different arguments = different cache entries

@cache.memoize(timeout=60)
def get_user_posts(user_id, page=1):
    # Called with user_id=1 -> stored as separate entry from user_id=2
    return Post.query.filter_by(user_id=user_id)\
        .order_by(Post.created_at.desc())\
        .paginate(page=page, per_page=10)


@cache.memoize(timeout=300)
def get_category_posts(category_slug):
    category = Category.query.filter_by(slug=category_slug).first_or_404()
    return Post.query.filter_by(category=category)\
        .order_by(Post.created_at.desc())\
        .limit(20).all()


# Usage in routes:
@app.route("/user/<int:user_id>/posts")
def user_posts(user_id):
    posts = get_user_posts(user_id)   # cached per user_id
    return render_template("user_posts.html", posts=posts)


# Invalidating memoized results when data changes:
@app.route("/post/create", methods=["POST"])
@login_required
def create_post():
    post = Post(...)
    db.session.add(post)
    db.session.commit()
    cache.delete_memoized(get_user_posts, current_user.id)   # clear this user's cache
    return redirect(url_for("main.index"))
"""

print("Function-level memoization:")
print(memoize_code)
print()
print("Key difference from @cached:")
print("  @cached:   one cache entry per URL (same for all users)")
print("  @memoize:  one cache entry per unique set of arguments")


### Cache Invalidation

> *"Cache invalidation is one of the two hard problems in computer science (the other is naming things)."* — Phil Karlton

The challenge: if cached data is stale, users see wrong information; if you invalidate too aggressively (clearing everything on every write), you eliminate all performance benefits. Good cache invalidation is precise, timely, and always paired with the code that causes the change.

**Three strategies — from most to least precise:**

**1. Explicit delete** — call `cache.delete()` or `cache.delete_memoized()` exactly when the underlying data changes.
- Most precise: only the affected entries are cleared; unrelated entries remain warm
- Requires discipline: every code path that modifies data must also invalidate the cache
- Risk: if you forget to invalidate at any write site, stale data persists until TTL expiry

**2. TTL expiry** — set a short enough TTL that stale data expires naturally, without any manual invalidation code.
- Simplest approach: no explicit invalidation logic required
- Trade-off: stale data is always possible for up to TTL seconds after a write
- Best for: data that changes infrequently and where short-term staleness is acceptable (e.g. a "top 10 posts" list cached for 5 minutes)

**3. `cache.clear()`** — the nuclear option. Clears *every* entry in the cache at once.
- Use after bulk data migrations or deployments where many cached values may be stale simultaneously
- Causes a **thundering herd**: all subsequent requests become cache misses at once, creating a sudden spike in database load. Use with care.

**The golden rule:** always invalidate in the **same code path** as the database write. If your `update_post()` function commits to the DB, call `cache.delete_memoized()` in the same function — never in a separate place where it might be forgotten or skipped:

```python
def update_post(post_id, new_title, new_body):
    post = Post.query.get(post_id)
    post.title = new_title
    post.body = new_body
    db.session.commit()                           # 1. write to DB
    cache.delete_memoized(get_post, post_id)      # 2. immediately invalidate cache
    cache.delete("view//posts")                   # 3. also clear the listing page
```

Stale cache is worse than no cache — users see outdated data. Always invalidate (clear) cached values when the underlying data changes:

In [ ]:

invalidation_code = """
# Cache invalidation -- the hardest part

# 1. Delete a specific cached route
cache.delete("view//popular-posts")

# 2. Delete a memoized function call by arguments
cache.delete_memoized(get_user_posts, user_id=42)

# 3. Delete ALL cached memoized calls for a function
cache.delete_memoized(get_user_posts)

# 4. Nuclear option: clear the entire cache
cache.clear()

# Pattern: invalidate on write operations
@posts.route("/post/<int:id>/edit", methods=["POST"])
@login_required
def edit_post(id):
    post = Post.query.get_or_404(id)
    post.title = form.title.data
    post.body  = form.body.data
    db.session.commit()

    # Invalidate caches that contain this post
    cache.delete("view//")               # homepage
    cache.delete("view//popular-posts")  # popular posts list
    cache.delete_memoized(get_user_posts, post.author_id)  # author's posts

    return redirect(url_for("posts.view_post", id=id))
"""

print("Cache invalidation strategies:")
print(invalidation_code)
print()
print("The three hard problems in computer science:")
print("  1. Naming things")
print("  2. Cache invalidation")
print("  3. Off-by-one errors")
print()
print("Tip: prefer short TTLs (60-300s) over perfect invalidation.")
print("Stale data for 5 minutes is usually acceptable.")
print("Manual invalidation is error-prone and easy to forget.")


## 🔬 Deep Dive

### Rate Limiting with Flask-Limiter

**Rate limiting** controls how many requests a client can make within a time window. Without it, a single misbehaving client (or attacker) can exhaust your server resources, overwhelm your database, or brute-force your login endpoint by trying thousands of passwords per second.

#### How Rate Limiting Algorithms Work

**Token Bucket** — the most common approach:
Imagine a bucket that automatically refills with tokens at a constant rate (e.g. 1 token per second, up to a maximum of 60). Each incoming request consumes one token. If the bucket is empty, the request is rejected with `429 Too Many Requests`. The bucket model allows short *bursts* of traffic (up to the full bucket capacity) while enforcing a sustained long-term rate.

```text
Bucket capacity:  60 tokens
Refill rate:      1 token / second
Effect:           allows up to 60 requests in a burst,
                  then at most 1 new request every second thereafter
```

**Sliding Window** — more precise, prevents burst-boundary abuse:
Instead of a refill bucket, the server tracks the **actual timestamps** of the last N requests. A new request is allowed only if fewer than the limit occurred within the *last* window. This prevents the "double-burst" problem: with a fixed window a client could make 60 requests at 11:59:59 and 60 more at 12:00:00 — 120 requests in 2 seconds. Sliding window prevents this because the window always looks back exactly 60 seconds from *now*.

#### Flask-Limiter Key Functions

By default, Flask-Limiter identifies clients by their **IP address** (`get_remote_address`). You can change this to limit per authenticated user:

```python
from flask_limiter.util import get_remote_address

# Default: limit by client IP address
limiter = Limiter(app, key_func=get_remote_address)

# Per authenticated user (fairer for shared IPs like offices or universities):
limiter = Limiter(app, key_func=lambda: str(current_user.id))
```

Per-user limits are fairer for legitimate users behind shared IPs and more effective against authenticated API abuse.

#### Dynamic Limits Per Route

Different endpoints need different thresholds:

```python
@app.route("/login", methods=["POST"])
@limiter.limit("10 per minute")       # brute-force protection — strict
def login(): ...

@app.route("/register", methods=["POST"])
@limiter.limit("5 per hour")          # spam prevention — very strict
def register(): ...

@app.route("/api/data")
@limiter.limit("100 per minute")      # API endpoint — more generous
def api_data(): ...
```

#### ⚠️ Redis is Required for Multi-Worker Deployments

Without a shared backend, each Gunicorn worker maintains its own **separate in-memory counter**. With a `"10 per minute"` limit and 4 workers, a client could make 10 × 4 = **40 requests per minute** before any worker's counter triggers a rejection.

```python
app.config["RATELIMIT_STORAGE_URI"] = "redis://localhost:6379"
```

With Redis, all workers share one atomic counter — the limit is enforced correctly across the entire application, regardless of which worker handles each request.

#### 429 Too Many Requests

When a client exceeds the limit, Flask-Limiter automatically returns an HTTP `429 Too Many Requests` response with a `Retry-After` header. You can customise the error handler:

```python
@app.errorhandler(429)
def ratelimit_handler(e):
    return {"error": "Too many requests", "retry_after": str(e.description)}, 429
```

Rate limiting restricts how many requests a client can make in a time window. It protects against brute-force attacks, API abuse, and accidental runaway clients:

In [ ]:

rate_limiting_code = """
# Install: pip install Flask-Limiter
from flask_limiter import Limiter
from flask_limiter.util import get_remote_address

limiter = Limiter(
    app,
    key_func=get_remote_address,   # limit per IP address
    default_limits=["200 per day", "50 per hour"]  # applies to all routes
)

# Override for specific routes:
@app.route("/api/send-email")
@limiter.limit("5 per minute")       # strict limit on expensive operations
def send_email():
    send_transactional_email(...)
    return jsonify({"status": "sent"})


@app.route("/auth/login", methods=["POST"])
@limiter.limit("10 per minute")      # prevent brute force attacks
def login():
    ...


@app.route("/api/search")
@limiter.limit("30 per minute")      # allow reasonable usage
def search():
    ...


# Exempting trusted routes:
@app.route("/health")
@limiter.exempt                       # no limit on health checks
def health():
    return "OK"
"""

print("Rate limiting:")
print(rate_limiting_code)
print()
print("When rate limit exceeded: Flask-Limiter returns 429 Too Many Requests")
print()
print("Key functions to rate-limit in any app:")
limits = [
    ("/auth/login",         "10 per minute",  "Prevent brute force"),
    ("/auth/register",      "5 per minute",   "Prevent mass account creation"),
    ("/api/send-email",     "5 per hour",     "Expensive operation"),
    ("/api/send-sms",       "3 per hour",     "Cost control"),
    ("/api/search",         "60 per minute",  "DB-intensive"),
    ("/api/",               "1000 per hour",  "General API protection"),
]
for route, limit, reason in limits:
    print(f"  {route:<25} {limit:<20} {reason}")


### The N+1 Query Problem

The N+1 problem is a performance trap where a loop that fetches N objects triggers N additional queries — one per object — instead of a single JOIN. It is the most common database performance issue in ORM-based applications, and it can be invisible until your data grows.

#### Concrete Counting Example

Suppose you have 100 blog posts and you want to display each post with its author's name:

```text
Query 1:     SELECT * FROM posts;                        -- 1 query: all 100 posts
Query 2:     SELECT * FROM users WHERE id = 1;           -- author of post 1
Query 3:     SELECT * FROM users WHERE id = 7;           -- author of post 2
Query 4:     SELECT * FROM users WHERE id = 3;           -- author of post 3
...
Query 101:   SELECT * FROM users WHERE id = 42;          -- author of post 100
```

**1 + 100 = 101 queries** for a single page load. With 1,000 posts: 1,001 queries. With 10,000 posts: 10,001 queries. This scales **linearly** with your data size — the more successful your app becomes, the more painful this gets.

#### Why ORMs Make N+1 Easy to Create Accidentally

SQLAlchemy's default **lazy loading** fetches related objects on first access. The code looks completely clean and reasonable:

```python
posts = Post.query.all()        # 1 query — fetches all posts
for post in posts:
    print(post.author.username) # silently fires a NEW SELECT query every iteration!
```

The `post.author` attribute access fires a `SELECT * FROM users WHERE id = ?` query each time, because SQLAlchemy didn't know you'd need the authors when it ran the original posts query. There is no warning, no error — it works correctly but hammers your database.

#### The Fix: Eager Loading with `joinedload()`

`joinedload()` tells SQLAlchemy to fetch posts *and* their authors in a **single SQL JOIN** — one round trip to the database:

```python
from sqlalchemy.orm import joinedload

posts = Post.query.options(joinedload(Post.author)).all()
# Executes: SELECT posts.*, users.*
#           FROM posts JOIN users ON posts.author_id = users.id

for post in posts:
    print(post.author.username)   # no extra query — author already in memory
```

**1 query total**, regardless of the number of posts.

#### Alternative: `subqueryload()`

`subqueryload()` runs a second optimised query instead of a JOIN. This avoids row-multiplication that JOINs produce on one-to-many relationships:

```python
from sqlalchemy.orm import subqueryload

# Fetching each author with ALL their posts (one-to-many):
authors = Author.query.options(subqueryload(Author.posts)).all()
# Executes: SELECT * FROM authors;
#           SELECT * FROM posts WHERE posts.author_id IN (1, 7, 3, 42, ...);
```

**When to use which:**

| Strategy | Best for | Why |
|---|---|---|
| `joinedload` | Many-to-one (post → author) | One joined row per post — compact result set |
| `subqueryload` | One-to-many (author → all posts) | Avoids row duplication from cartesian-product JOINs on large sets |

In [ ]:

# The most common database performance problem in Flask/SQLAlchemy

n_plus_1_code = """
# BAD: N+1 queries -- this is the most common perf mistake

posts = Post.query.all()          # 1 query to get all posts

for post in posts:
    print(post.author.username)   # 1 EXTRA QUERY PER POST to load the author!
    # If you have 100 posts: 1 + 100 = 101 queries total!
    # Flask-SQLAlchemy lazy-loads relationships by default
"""

good_code = """
# GOOD: eager loading -- 1 query with a JOIN

from sqlalchemy.orm import joinedload

posts = Post.query.options(joinedload(Post.author)).all()
# 1 query: SELECT posts.*, users.* FROM posts JOIN users ON ...

for post in posts:
    print(post.author.username)   # no extra query! already loaded
"""

detect = """
# Detecting N+1 queries in development:
# Install: pip install flask-debugtoolbar
from flask_debugtoolbar import DebugToolbarExtension

app.config["DEBUG_TB_ENABLED"] = True
app.config["SECRET_KEY"] = "dev-key"
toolbar = DebugToolbarExtension(app)

# The toolbar shows SQL queries per request in the browser.
# 100+ queries on a single page = almost certainly an N+1 problem.
"""

print("The N+1 Problem:")
print(n_plus_1_code)
print("The Fix:")
print(good_code)
print("Detection:")
print(detect)


### Database Indexes

A database index is a data structure that lets the database find rows matching a condition without scanning every row. Without indexes, queries on large tables become progressively slower as data grows — and the degradation is linear.

#### What a Table Scan Is

Without an index, the database performs a **full table scan**: it reads *every single row* in the table and checks whether it matches the condition. On a 1-million-row `users` table, a query for `WHERE email = 'alice@example.com'` reads **1,000,000 rows**, compares each one, and returns the single match. This is O(n) — doubling the table size doubles the query time.

#### What a B-Tree Index Is

The default index type in PostgreSQL (and most databases) is a **B-tree (balanced tree)**. Conceptually, it is a sorted tree structure where each internal node acts as a signpost pointing left (smaller values) or right (larger values), and each leaf node contains a pointer to the actual database row.

- **Lookup complexity: O(log n)** instead of O(n)
- A 1-million-row lookup goes from ~1,000,000 row comparisons to **~20 comparisons** (log₂(1,000,000) ≈ 20)
- The tree stays balanced automatically as rows are inserted, updated, and deleted

#### When to Add Indexes

| Column type | Why index it |
|---|---|
| Foreign key columns (`author_id`, `post_id`, etc.) | JOINs and FK lookups are extremely common; unindexed FKs cause full scans on every JOIN |
| Columns in `WHERE` clauses (`email`, `slug`, `status`) | Filters use the index instead of scanning all rows |
| Columns in `ORDER BY` clauses (`created_at`, `score`) | Sorting uses the pre-sorted index — eliminates an explicit sort step |
| Columns with `UNIQUE` constraints | The DB enforces uniqueness via the index; no extra cost |

#### Compound Indexes

When you frequently filter by two columns *together*, a compound index is more efficient than two separate single-column indexes:

```python
# SQLAlchemy model — compound index on (author_id, created_at):
__table_args__ = (
    Index('idx_author_created', 'author_id', 'created_at'),
)
# Efficiently answers: posts by a specific author, sorted by date
# SQL: SELECT * FROM posts WHERE author_id = 42 ORDER BY created_at DESC;
```

**Column order matters**: `(author_id, created_at)` is efficient for filtering by `author_id` alone *or* by both columns together, but is not useful for filtering by `created_at` alone (the database can't use the middle of a tree without the leading column).

#### Diagnosing with `EXPLAIN ANALYZE`

Prepend `EXPLAIN ANALYZE` to any PostgreSQL query to see the **query execution plan** — exactly how the database runs it, what it scans, which indexes it uses, and how long each step takes:

```sql
EXPLAIN ANALYZE
  SELECT * FROM posts WHERE author_id = 42 ORDER BY created_at DESC;

-- Good output (index used):
--   Index Scan using idx_author_created on posts  (cost=0.43..8.45 rows=5)
--   Execution time: 0.1 ms

-- Bad output (full scan):
--   Seq Scan on posts  (cost=0.00..24310.00 rows=1000000)
--   Execution time: 4820.3 ms
```

**`Seq Scan`** = sequential full table scan (bad for large tables, good only for very small ones). **`Index Scan`** or **`Index Only Scan`** = the query used an index (good).

#### ⚖️ The Index Trade-off

Indexes are **not free**: every `INSERT`, `UPDATE`, and `DELETE` must also update every index on that table. A table with 10 indexes takes roughly 10× more write overhead than a table with none.

**Rule of thumb:** index columns you query or sort frequently; avoid indexing columns that change on almost every write on write-heavy tables. Don't index every column blindly — measure first with `EXPLAIN ANALYZE`, then add indexes only where the query plan shows a sequential scan on a large table.

In [ ]:

indexes_code = """
# Without an index, every query on a non-primary-key column does a FULL TABLE SCAN
# With 1 million rows: full scan = slow, index scan = fast

# SQLAlchemy model with indexes:
class Post(db.Model):
    id         = db.Column(db.Integer, primary_key=True)   # auto-indexed
    title      = db.Column(db.String(200), nullable=False)
    body       = db.Column(db.Text, nullable=False)
    created_at = db.Column(db.DateTime, default=datetime.utcnow, index=True)
    #                                                              ^^^^^^^^^
    #                                               index on the column itself

    author_id  = db.Column(db.Integer, db.ForeignKey("users.id"), index=True)
    #                                                               ^^^^^^^^^
    #                           index FK columns -- used in JOINs constantly

    slug       = db.Column(db.String(200), unique=True, index=True)
    #                                      unique=True auto-creates unique index


# Composite index (when querying by multiple columns together):
class Post(db.Model):
    ...
    __table_args__ = (
        db.Index("idx_post_author_created", "author_id", "created_at"),
        # Speeds up: Post.query.filter_by(author_id=x).order_by(Post.created_at)
    )
"""

when_to_index = [
    ("Primary keys",        "Auto-indexed"),
    ("Foreign keys",        "Always index -- used in every JOIN"),
    ("Columns in WHERE",    "Index if queried frequently"),
    ("Columns in ORDER BY", "Index if sorted frequently"),
    ("Unique columns",      "unique=True auto-creates an index"),
    ("columns in LIKE",     "Special full-text indexes needed"),
]

print("Database indexing:")
print(indexes_code)
print()
print("When to add an index:")
for col_type, advice in when_to_index:
    print(f"  {col_type:<25} {advice}")
print()
print("Cost of indexes:")
print("  SELECTs become faster (index lookup vs full scan)")
print("  INSERTs/UPDATEs become slightly slower (index must be updated)")
print("  Each index uses disk space")
print("  Rule of thumb: index columns used in WHERE or JOIN, not all columns")


### ⚖️ SimpleCache vs RedisCache

#### Understanding Deployment Context First

Before comparing backends, understand the deployment model. A single-process development server runs **one Python process** — all requests are handled by that one process, and its memory is shared across all requests.

A production app with 4 Gunicorn workers runs **4 separate Python processes**, each with its own isolated memory space. SimpleCache stores entries in the process's own memory. In a 4-worker deployment, each worker has its own *isolated* SimpleCache — worker 1's cached value is invisible to workers 2, 3, and 4. If worker 2 handles your next request, it has a cold cache even if worker 1 computed and cached the same value 2 seconds ago. In the worst case, effective cache hit rate approaches zero because requests are load-balanced round-robin across workers.

**RedisCache solves this**: all workers connect to one shared Redis server. Worker 1 writes a value, and all other workers can read it immediately. Cache hit rate reflects your actual TTL and traffic patterns, not process assignment luck.

Choosing a cache backend depends on your deployment requirements:

In [ ]:

comparison = """
+-------------------+-----------------------------+-------------------------------------+
| Aspect            | SimpleCache                 | RedisCache                          |
+-------------------+-----------------------------+-------------------------------------+
| Setup             | Zero -- built-in Python     | Requires Redis server               |
| Shared across     | Single process only         | All Gunicorn workers + servers      |
| Survives restart  | No -- lost on app restart   | Yes -- persisted by Redis           |
| Max size          | Configurable, in RAM        | Configurable, in RAM (with AOF/RDB) |
| Performance       | Extremely fast (in-process) | Very fast (sub-millisecond network) |
| Best for          | Development, single server  | Production, multiple workers        |
| Distributed apps  | Not possible                | Yes -- shared across servers        |
+-------------------+-----------------------------+-------------------------------------+
"""

migration_path = """
# config.py -- switch caches based on environment
class DevelopmentConfig:
    CACHE_TYPE = "SimpleCache"      # no Redis needed in dev

class ProductionConfig:
    CACHE_TYPE = "RedisCache"
    CACHE_REDIS_URL = os.environ["REDIS_URL"]   # set by hosting provider
"""

print("SimpleCache vs RedisCache:")
print(comparison)
print()
print("Environment-based configuration:")
print(migration_path)
print()
print("Practical advice:")
print("  Development:  SimpleCache is fine, keep it simple")
print("  Production (1 server, 1 worker): SimpleCache works")
print("  Production (Gunicorn with -w 4): use RedisCache!")
print("  Each Gunicorn worker is a separate process.")
print("  SimpleCache is per-process -- 4 workers = 4 independent caches!")


### ⚖️ `@cached` vs `@memoize`

**The key question is: what is the cache key?** For `@cache.cached()`, it is the URL path. For `@cache.memoize()`, it is the function name plus all of its arguments. This determines what "one cache entry" means — and therefore what "invalidating one entry" means.

- `@cached` → one entry per unique URL → invalidate with `cache.delete("view//path")`
- `@memoize` → one entry per unique `(function, arg1, arg2, …)` combination → invalidate with `cache.delete_memoized(fn, arg1, arg2)`

A practical consequence: if `get_user_posts(user_id=42)` is called from three different routes, `@cache.memoize()` caches it *once* and all three routes share the cached result. With `@cache.cached()` on each route, you'd have three separate cache entries for the same underlying data.

Both decorators cache return values, but they serve different purposes:

In [ ]:

when_to_use = """
@cache.cached()    -- route-level, one entry per URL
    Use when: the entire page response is the same for all users
    Example: /popular-posts, /about, /category/python
    Cache key: the URL path

@cache.memoize()   -- function-level, one entry per argument combination
    Use when: a function's result varies by its arguments
    Example: get_user_posts(user_id), get_category(slug)
    Cache key: function name + arguments hash

    ALSO use memoize when the same function is called from multiple routes
    -- you get reuse across routes automatically.
"""

decision_tree = """
Decision tree:
  Does the response depend on the CURRENT USER?
    Yes -> Don't cache at route level (or vary by user)
    No  -> @cache.cached() on the route

  Is this a function called from multiple places with different args?
    Yes -> @cache.memoize() on the function
    No  -> @cache.cached() on the route might be simpler

  Does the function have complex arguments (objects, lists)?
    Yes -> @cache.memoize() requires make_cache_key override
    No  -> @cache.memoize() works out of the box
"""

print("When to use cached vs memoize:")
print(when_to_use)
print(decision_tree)


## 🧪 What If?

### What if you cache user-specific data without varying by user?

Caching a view that returns personalised data without varying the cache key by user is a serious bug — every user will see the same cached response (potentially someone else's data).

#### The Fix: Vary the Cache Key by User

Use the `key_prefix` parameter to make the cache key unique per authenticated user:

```python
# Option 1: lambda key_prefix (simple, recommended)
@app.route("/dashboard")
@login_required
@cache.cached(timeout=60, key_prefix=lambda: f"dashboard_{current_user.id}")
def dashboard():
    # Each user gets their own cache entry: "dashboard_42", "dashboard_99", etc.
    return render_template("dashboard.html", data=get_user_data(current_user.id))

# Option 2: make_cache_key callable (full control over key construction)
def user_dashboard_key():
    return f"dashboard_{current_user.id}_{request.path}"

@app.route("/dashboard")
@login_required
@cache.cached(timeout=60, make_cache_key=user_dashboard_key)
def dashboard(): ...
```

**Important:** when you vary the cache key by user, cache invalidation must also be per-user. Use `cache.delete(f"dashboard_{user_id}")` when that specific user's data changes — don't call `cache.clear()` and evict everyone's cached dashboards for one user's update.

Caching a view that returns personalised data without varying the cache key by user is a serious bug — every user will see the same cached response (potentially someone else's data):

In [ ]:

disaster_scenario = """
DISASTER SCENARIO:
-------------------
@app.route("/dashboard")
@login_required
@cache.cached(timeout=300)   # WRONG! No user variation
def dashboard():
    # Shows user's personal stats, posts, notifications
    return render_template("dashboard.html", user=current_user)

What happens:
  User Alice (id=1) visits /dashboard
  -> Flask runs the function, renders Alice's dashboard
  -> Response is cached under key "view//dashboard"

  User Bob (id=2) visits /dashboard (within 5 minutes)
  -> Cache HIT! Returns Alice's dashboard!
  -> Bob sees Alice's personal data, posts, and notifications
  -> CATASTROPHIC privacy breach

FIX: vary the cache key by user
@app.route("/dashboard")
@login_required
@cache.cached(timeout=300, key_prefix=lambda: f"view//dashboard/user/{current_user.id}")
def dashboard():
    return render_template("dashboard.html", user=current_user)

OR: Simply don't cache user-specific pages at route level.
Cache only the expensive sub-components with @memoize.
"""

print(disaster_scenario)
print("Rule: ALWAYS ask 'Is this response the same for every user?'")
print("If NO -> do not use @cache.cached() without user-specific key")


### What if cache is never invalidated? What if cache goes down?

Two common caching failures with very different consequences:


In [ ]:

print("=== Never Invalidating the Cache ===")
stale_problem = """
Scenario: You cache the list of categories for 24 hours.
An admin adds a new category "Machine Learning".
For 24 hours, no user sees it -- they still see the stale list.
The admin thinks the site is broken.

Solution spectrum:
  1. Short TTL (60-300s): accept slightly stale data, no invalidation needed
  2. Manual invalidation: call cache.clear() or cache.delete() after writes
  3. Event-driven: use signals (blinker) to auto-invalidate on model changes
  4. No caching for data that must be real-time: user notifications, stock prices
"""
print(stale_problem)

print()
print("=== Cache Server Goes Down ===")
cache_failure = """
If Redis is down and your code assumes cache always works:
    posts = cache.get("popular_posts")  # returns None (not an error)
    # If you forget to handle None: TypeError: 'NoneType' is not iterable

Best practice: always write cache-tolerant code
    def get_popular_posts():
        posts = cache.get("popular_posts")
        if posts is None:                           # cache miss OR cache down
            posts = Post.query.order_by(...).all()
            try:
                cache.set("popular_posts", posts)   # best-effort cache write
            except Exception:
                pass                                # continue without cache
        return posts

Flask-Caching's @cache.cached() handles this automatically.
It falls back to calling the function if cache is unavailable.
This is another reason to use decorators over manual cache.get/set.
"""
print(cache_failure)


## 🚀 Real-World Project Link

In our **Blog Application**, caching and performance are applied strategically:

```python
# Cached routes (same for all users):
@app.route("/")
@cache.cached(timeout=300)         # 5 minutes -- homepage with latest/popular posts
def index(): ...

@app.route("/category/<slug>")
@cache.cached(timeout=600)         # 10 minutes -- category post list
def category(slug): ...

# Memoized functions (vary by argument):
@cache.memoize(timeout=120)
def get_popular_posts(limit=10): ...   # invalidated when a post's view_count changes

@cache.memoize(timeout=300)
def get_tag_posts(tag_slug): ...       # invalidated when posts are tagged/untagged
```

**Rate limiting:**
```python
@limiter.limit("10 per minute")   # login -- brute force protection
@limiter.limit("5 per minute")    # register -- spam prevention
@limiter.limit("30 per minute")   # comment API -- spam prevention
```

**Database optimisation:**
- All FK columns have indexes
- `joinedload(Post.author)` on list queries -- eliminates N+1
- Composite index on `(author_id, created_at)` for author's post history

> ❓ **What is a decorator?** A decorator is a function that wraps another function to add behaviour before or after it runs. `@app.route('/')` is shorthand for `index = app.route('/')(index)` — it registers your view function with Flask's URL map without any explicit call.

## 📋 Chapter Summary & Cheat Sheet

### Flask-Caching Setup

```python
from flask_caching import Cache
cache = Cache(app, config={"CACHE_TYPE": "SimpleCache"})     # dev
cache = Cache(app, config={"CACHE_TYPE": "RedisCache",
    "CACHE_REDIS_URL": os.environ["REDIS_URL"]})             # prod
```
### Caching Decorators

```python
@app.route("/popular")
@cache.cached(timeout=300)           # 5 min, all users same response
def popular(): ...

@cache.memoize(timeout=60)
def get_user_posts(user_id): ...     # per-argument cache entry
```
### Cache Invalidation

```python
cache.delete("view//popular")               # specific route
cache.delete_memoized(get_user_posts, 42)   # specific function+args
cache.clear()                               # nuclear option
```
### Rate Limiting

```python
from flask_limiter import Limiter
limiter = Limiter(app, key_func=get_remote_address)

@limiter.limit("10 per minute")
def login(): ...
```
### N+1 Fix

```python
# BAD (N+1):
posts = Post.query.all()
for p in posts: print(p.author.username)  # 1 query per post!

# GOOD (1 query):
from sqlalchemy.orm import joinedload
posts = Post.query.options(joinedload(Post.author)).all()
```

### When to Cache

| Cache it | Don't cache it |
|----------|----------------|
| Same for all users | User-specific pages |
| Expensive DB queries | Real-time data |
| Rarely changes | Data that must be fresh |
| Public pages | Authenticated dashboards |

## 💪 Practice Prompts

**1. Add Caching to a Flask App**
Add Flask-Caching to an existing app using `SimpleCache`. Cache your homepage for 2 minutes and your most-viewed posts page for 5 minutes. Verify it works by adding a `print()` statement inside the cached function -- it should print only on cache misses, not on every request. Then add cache invalidation: when a new post is created, delete the homepage and popular-posts cache.

**2. Fix an N+1 Query**
Create a route that displays 20 posts with their author names. First, implement it naively (lazy-loading, which creates N+1 queries). Install `flask-debugtoolbar` to count the queries. Then add `joinedload(Post.author)` and observe the query count drop from 21 to 1. Measure the response time difference.

**3. Implement Rate Limiting**
Add Flask-Limiter to your app. Rate-limit your login route to 5 attempts per minute, your registration route to 3 per hour, and any external API calls to 10 per minute. Write a test that verifies the 429 response is returned after exceeding the limit (use `with app.test_request_context()` to simulate multiple requests).

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../6.%20advanced_features/16.%20testing_flask_applications.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#8592; Chapter 16: Testing</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#128218; Table of Contents</a>
  <a href='18.%20deployment.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 18: Deployment &#8594;</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../6. advanced_features/16. testing_flask_applications.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='18. deployment.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>